In [8]:
import pandas as pd
from pathlib import Path

In [9]:
# Set the root directory
root_dir = Path('/Users/liuq13/SPCD_data/beh_raw')
outBaseDir = Path('/Users/liuq13/SPCD_data/beh_processed')

In [ ]:
# This script creates a reference table for all odors used in the SPCD task.
# It includes odor index, full name, short name, MFC line assignments, and category.
# The category column classifies odors as 'savory' (idx 1–5), 'sweet' (idx 6–10), or 'neutral' (idx 0).
# The final table is saved as 'odor_reference.csv' for use in analysis or lookup.

odor_data = {
    'idx': list(range(11)),
    'odor_full_name': [
        'Clean air', 'Potato chip', 'Garlic', 'Cheddar cheese',
        'Margherita pizza', 'Canned tuna', 'Strawberry', 'Chocolate',
        'Apple cinnamon', 'Peach', 'Vanilla'
    ],
    'odor_name': [
        'CLEN', 'POCH', 'GARL', 'CHEZ', 'PIZA', 'TUNA',
        'STRA', 'CHOC', 'CINN', 'PEAC', 'VANI'
    ],
    'line1': [6, 1, 2, 3, 4, 5, 6, 6, 6, 6, 6],
    'line2': [12, 12, 12, 12, 12, 12, 7, 8, 9, 10, 11]
}

# Create DataFrame
odor_df = pd.DataFrame(odor_data)

# Assign odor category
def get_category(idx):
    if idx == 0:
        return 'neutral'
    elif 1 <= idx <= 5:
        return 'savory'
    elif 6 <= idx <= 10:
        return 'sweet'

odor_df['category'] = odor_df['idx'].apply(get_category)

# Save to CSV
odor_out_file = outBaseDir / 'odor_reference.csv'
odor_df.to_csv(odor_out_file, index=False)

print("✅ Odor reference table created and saved as 'odor_reference.csv'")


In [12]:
# This script compiles odor selection info across all subjects in the SPCD task.
# It reads each subject's SelectedOdors.xlsx file, extracts O1 and O2 info,
# and uses the odor reference table to determine the devaluation type based on O2's category.
# The final output is a CSV file with subject ID, selected odors, and devaluation type.

odor_ref_path = outBaseDir / 'odor_reference.csv'

# Load odor reference
odor_ref = pd.read_csv(odor_ref_path)

# Initialize a list to collect subject-level odor info
all_data = []

# Loop through each subject folder
for subject_folder in root_dir.glob('sub-*'):
    odor_file = subject_folder / 'SelectedOdors.xlsx'
    
    if odor_file.exists():
        df = pd.read_excel(odor_file)
        df['subject'] = subject_folder.name
        all_data.append(df)
    else:
        print(f"⚠️ Missing SelectedOdors.xlsx in {subject_folder.name}")

# Concatenate and sort
if all_data:
    result_df = pd.concat(all_data, ignore_index=True)
    result_df = result_df[['subject', 'odor_use', 'odor_idx', 'odor_name']]

    # Sort by subject numerically
    result_df['subject_num'] = result_df['subject'].str.extract(r'(\d+)').astype(int)
    result_df = result_df.sort_values('subject_num').drop(columns='subject_num')

    # Pivot to get O2 row per subject for devaluation type
    o2_df = result_df[result_df['odor_use'] == 'O2'][['subject', 'odor_idx']].copy()
    o2_df = o2_df.rename(columns={'odor_idx': 'O2_odor_idx'})

    # Merge with odor reference to get category of O2
    o2_df = o2_df.merge(odor_ref[['idx', 'category']], left_on='O2_odor_idx', right_on='idx', how='left')
    o2_df = o2_df.rename(columns={'category': 'devaluation_type'}).drop(columns='idx')

    # Merge devaluation_type back to result_df
    result_df = result_df.merge(o2_df[['subject', 'devaluation_type']], on='subject', how='left')

    # Save to CSV
    output_file = outBaseDir / 'compiled_selected_odors.csv'
    result_df.to_csv(output_file, index=False)
    print(f"✅ Saved compiled data to {output_file}")
else:
    print("❌ No data collected. Check folder structure.")

✅ Saved compiled data to /Users/liuq13/SPCD_data/beh_processed/compiled_selected_odors.csv


In [17]:
# This script checks for consistency in devaluation type across two sources:
# - 'compiled_selected_odors.csv': derives devaluation type from the odor marked as O2
# - 'subject_info.csv': infers devaluation type from the 'feeding' column ('w' = sweet, 'a' = savory)
# It compares the two for each subject and reports any mismatches.

# Define paths
task_info_dir = Path('/Users/liuq13/SPCD_data/task_info')
odor_file = task_info_dir / 'compiled_selected_odors.csv'
subject_file = task_info_dir / 'subject_info.csv'

# Load data
odor_df = pd.read_csv(odor_file)
subject_df = pd.read_csv(subject_file)

# Get devaluation_type from compiled_selected_odors.csv (only O2 rows)
odor_deval_df = odor_df[odor_df['odor_use'] == 'O2'][['subject', 'devaluation_type']].copy()
odor_deval_df = odor_deval_df.rename(columns={'devaluation_type': 'deval_from_odor_file'})

# Get devaluation_type from subject_info.csv using 'feeding'
def feeding_to_deval(feeding):
    if feeding == 'w':
        return 'sweet'
    elif feeding == 'a':
        return 'savory'
    else:
        return 'unknown'

subject_df['deval_from_subject_file'] = subject_df['feeding'].apply(feeding_to_deval)

# Keep only relevant columns
subject_deval_df = subject_df[['subject', 'deval_from_subject_file']]

# Merge and compare
merged_df = pd.merge(odor_deval_df, subject_deval_df, on='subject', how='inner')
merged_df['match'] = merged_df['deval_from_odor_file'] == merged_df['deval_from_subject_file']

# Print mismatches
mismatches = merged_df[~merged_df['match']]

if not mismatches.empty:
    print("❗ Mismatches found between odor and subject files:\n")
    print(mismatches)
else:
    print("✅ All subjects have matching devaluation type across files.")


❗ Mismatches found between odor and subject files:

  subject deval_from_odor_file deval_from_subject_file  match
9  sub-14               savory                   sweet  False


In [ ]:
# for this file I'll leave it as it is, but for other data analyses
# I will do some extra processing to reflect this difference, see code for sub-14 for doing this